In [1]:
PREFIX="https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed"
!wget $PREFIX/download.py
!wget $PREFIX/embedder.py

--2026-06-29 11:00:21--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed/download.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1376 (1.3K) [text/plain]
Saving to: ‘download.py.8’

download.py.8       100%[===================>]   1.34K  --.-KB/s    in 0s      

2026-06-29 11:00:22 (38.9 MB/s) - ‘download.py.8’ saved [1376/1376]

--2026-06-29 11:00:22--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed/embedder.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

/mnt/c/Users/xie_j/Documents/llm-zoomcamp-hw2/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12090). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [3]:
import sentence_transformers
print(sentence_transformers.__version__) 
# 0.2.5.1 is outdated -->
# using this kernel instead: .venv/bin/python -m ipykernel install --user --name llm-hw2 --display-name "LLM HW2"

5.6.0


In [6]:
"""
Q1. Embedding a query
"""

q = "How does approximate nearest neighbor search work?"
v = model.encode(q)
v.shape

(384,)

In [7]:
# The embedder returns a vector of 384 numbers. What's the first value (v[0])?

v[0]

np.float32(-0.020582011)

In [8]:
# Loading the data:

from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [12]:
"""
Q2. Cosine similarity

Take the page 02-vector-search/lessons/07-sqlitesearch-vector.md, embed its content, and compute the cosine similarity with the query vector from Q1. 
What do you get?
"""

# Each document is a dictionary with a filename and content, and there are 72 pages.

doc = [d for d in documents if "07-sqlitesearch-vector" in d["filename"]][0]
print(doc["filename"])

02-vector-search/lessons/07-sqlitesearch-vector.md


In [19]:
v_lesson_07 = model.encode(doc["content"])
v.dot(v_lesson_07)

np.float32(0.45681328)

In [24]:
import json
documents[10]

{'content': "# Agents\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=6uG4_Ivv60E&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn Part 1 of this module we built RAG pipelines.\n\nEvery pipeline we wrote followed the same flow:\n\n- search the FAQ,\n- build a prompt with the results,\n- send it to the LLM.\n\nThis returns good answers when the user's query matches the documents.\nThe search finds the right entry, the LLM reads it, and you get a\nhelpful reply.\n\nOften, though, the search returns nothing useful.\n\n- Maybe the user made a typo.\n- Maybe they asked the question in an unusual way.\n- Maybe they need information from two different searches.\n\nWe use lexical search here, so the search looks for an exact match.\nOne typo and it misses the entry it needed. In our pipeline there's\nno recovery. The search runs once, and if it returns garbage the LLM\ngets garbage. Our pipeline always does the same thing, no matter what.\n\nInstead of routing the user question str

In [26]:
"""
Q3. Chunking and search by hand

"""

from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

# We embed every chunk's content with encode_batch, stack the vectors into a matrix X, and score the Q1 query against all chunks:


In [35]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

texts = [chunk['content'] for chunk in chunks]

vectors = []
for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

X = np.array(vectors)
scores = X.dot(v)

idx = np.argmax(scores)
chunks[idx]['filename']

  0%|          | 0/6 [00:00<?, ?it/s]

'02-vector-search/lessons/07-sqlitesearch-vector.md'

In [36]:
# getting the highest indexes:

idx = np.argmax(scores)
idx, scores[idx]

(np.int64(94), np.float32(0.562441))

In [39]:
"""
Q4. Vector search with minsearch:

We've done vector search by hand, which is good for learning, but it's not what we do in practice. In practice we use libraries.

Let's use VectorSearch from minsearch and run a search for the following query:

What metric do we use to evaluate a search engine?

Which file is the filename of the first result?
"""

from minsearch import VectorSearch

q4 = "What metric do we use to evaluate a search engine?"
v4 = model.encode(q4)

vindex = VectorSearch(keyword_fields=['filename'])
vindex.fit(X, chunks)

results = vindex.search(v4, num_results=5)
results[0]['filename']

'04-evaluation/lessons/01-intro.md'

In [40]:
"""
Q5. Text search vs vector search
Vector search matches by meaning, keyword search by exact words.

Let's compare them. Index the same chunks with Index from minsearch. Use content as a text field.

Run both searches for this query:

How do I store vectors in PostgreSQL?

Take the top 5 results from each method. Which file shows up in the vector results but not in the text results?
"""

from minsearch import Index

# Text search index
text_index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)
text_index.fit(chunks)

q5 = "How do I store vectors in PostgreSQL?"
v5 = model.encode(q5)

# Vector results (top 5)
vector_results = vindex.search(v5, num_results=5)
vector_files = set(r['filename'] for r in vector_results)

# Text results (top 5)
text_results = text_index.search(q5, num_results=5)
text_files = set(r['filename'] for r in text_results)

# Files in vector but not in text
print(vector_files - text_files)

{'02-vector-search/lessons/08-pgvector.md'}


In [41]:
"""
Q6. Hybrid search
"""

def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

q6 = "How do I give the model access to tools?"
v6 = model.encode(q6)

vector_results = vindex.search(v6, num_results=10)
text_results = text_index.search(q6, num_results=10)

results = rrf([vector_results, text_results])
results[0]['filename']

'01-agentic-rag/lessons/14-agentic-loop.md'